# Landing on the vehicle in a curve: RL policy against the cascade

Compares two flights on the "real" drone (`UAV_model.slx`), same vehicle (`car_path.m`), same height policy:
- **cascade**: `platform3d_flight.csv`, written by `EINT_PLATFORM3D_eval.m` (x and y: position loop + angle loop);
- **RL**: `platform3d_rl_flight.csv`, written by `EINT_PLATFORM3D_RL_eval.m` (x and y: policy of `track_b_x.ipynb`).

Run both scripts in MATLAB first (with the same `V_CAR`, `R_TURN` in `car_path.m`). Outputs: a table of indicators,
a figure (saved in `../results/figures/compare_rl_cascade.png`) and a 3D animation with the two drones on the same vehicle
(`../results/figures/compare_rl_cascade.gif`).

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from IPython.display import HTML

RL_DIR = Path(r'C:\Users\ezabl\Downloads\Student_folder\Student_folder\rl_interface')   # change if needed
FILES = {'cascade': RL_DIR / 'platform3d_flight.csv', 'RL': RL_DIR / 'platform3d_rl_flight.csv'}
COLORS = {'cascade': 'tab:blue', 'RL': 'tab:red'}
FIG_DIR = Path('../results/figures'); FIG_DIR.mkdir(parents=True, exist_ok=True)
G = 9.81

def load(path):
    """One flight. Columns: t z vz az_prev x vx theta wy ax az y vy phi psi wx wz u1 u2 u3 u4 x_car y_car."""
    F = np.loadtxt(path, delimiter=',')
    f = dict(zip('t z vz az_prev x vx theta_meas wy ax az y vy phi psi wx wz u1 u2 u3 u4 x_car y_car'.split(), F.T))
    Ts = f['t'][1] - f['t'][0]
    # theta for the drawing: same complementary filter as the controllers (the measured theta is unusable)
    th = np.zeros_like(f['t'])
    for k in range(1, len(th)):
        th[k] = th[k - 1] + Ts * (f['wy'][k] + f['wy'][k - 1]) / 2
        if f['az'][k] + G > 2:
            th[k] = 0.98 * th[k] + 0.02 * np.arctan2(f['ax'][k], f['az'][k] + G)
    f['theta'] = th
    f['gap'] = np.hypot(f['x'] - f['x_car'], f['y'] - f['y_car'])       # horizontal distance drone - platform
    f['dx'] = (f['u3'] - f['u1']) / 2                                     # difference of command on the x axis
    f['dy'] = (f['u2'] - f['u4']) / 2                                     # ... on the y axis
    return f

def indicators(f):
    t, z, gap = f['t'], f['z'], f['gap']
    k = np.flatnonzero((t >= 5.5) & (z <= 0.02))
    k = k[0] if len(k) else len(t) - 1
    return {'contact time (s)': t[k],
            'distance at contact (cm)': 100 * gap[k],
            'vz at contact (m/s)': f['vz'][k],
            'max distance from 4 s (cm)': 100 * gap[t >= 4].max(),
            'mean distance last 2 s (cm)': 100 * gap[t >= t[-1] - 2].mean(),
            'max tilt theta / phi (rad)': max(np.abs(f['theta']).max(), np.abs(f['phi']).max()),
            'mean |change of delta| per step': np.mean(np.abs(np.diff(f['dx']))) + np.mean(np.abs(np.diff(f['dy'])))}

flights = {name: load(p) for name, p in FILES.items()}
ind = {name: indicators(f) for name, f in flights.items()}
print(f"{'':36s}" + ''.join(f'{n:>12s}' for n in flights))
for key in ind['cascade']:
    print(f'{key:36s}' + ''.join(f'{ind[n][key]:12.3f}' for n in flights))

**Figure**: distance to the platform, view from above, height, and the differences of command δ (the "work" of each
controller: a controller that zigzags makes the motors work hard).

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
for name, f in flights.items():
    c = COLORS[name]
    ax[0, 0].plot(f['t'], 100 * f['gap'], c, label=name)
    ax[0, 1].plot(f['x'], f['y'], c, label=name)
    ax[1, 0].plot(f['t'], f['z'], c, label=name)
    ax[1, 1].step(f['t'], f['dx'], c, where='post', label=f'{name}, x axis')
    ax[1, 1].step(f['t'], f['dy'], c, where='post', ls=':', label=f'{name}, y axis')
f = flights['cascade']
ax[0, 1].plot(f['x_car'], f['y_car'], 'k--', label='vehicle')
ax[0, 0].set(xlabel='t [s]', ylabel='distance to the platform [cm]', title='Horizontal distance drone - platform')
ax[0, 1].set(xlabel='x [m]', ylabel='y [m]', title='Seen from above'); ax[0, 1].axis('equal')
ax[1, 0].set(xlabel='t [s]', ylabel='z [m]', title='Height (same Track B policy)')
ax[1, 1].set(xlabel='t [s]', ylabel='delta', title='Differences of command (x: u3-u1, y: u2-u4, halved)')
for a in ax.flat:
    a.grid(alpha=0.3); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / 'compare_rl_cascade.png', dpi=150); plt.show()

**3D animation**: the two drones (red = RL, blue = cascade) above the same vehicle. Same drawing as
`platform_animation_3d.ipynb`; a vertical line links each drone to the level of the platform (the 3D perspective is misleading).

In [ ]:
Z_GROUND, ARM = -0.25, 0.12
CAR_L, CAR_W, PLAT = 0.9, 0.5, 0.4
f0 = flights['cascade']; t = f0['t']; Ts = t[1] - t[0]
n = min(len(f['t']) for f in flights.values())
vxc, vyc = np.gradient(f0['x_car'], Ts), np.gradient(f0['y_car'], Ts)
heading = np.where(np.hypot(vxc, vyc) > 1e-3, np.arctan2(vyc, vxc), 0.0)

def rot_z(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

def rotation(ph, th, ps):
    cp, sp, ct, st = np.cos(ph), np.sin(ph), np.cos(th), np.sin(th)
    return rot_z(ps) @ np.array([[ct, 0, st], [0, 1, 0], [-st, 0, ct]]) @ np.array([[1, 0, 0], [0, cp, -sp], [0, sp, cp]])

def box_faces(xc, yc, h):
    corners = np.array([[sx * CAR_L / 2, sy * CAR_W / 2] for sx, sy in [(-1, -1), (1, -1), (1, 1), (-1, 1)]]) @ rot_z(h)[:2, :2].T + [xc, yc]
    low = [(*c, Z_GROUND + 0.05) for c in corners]; high = [(*c, -0.02) for c in corners]
    return [low, high] + [[low[i], low[(i + 1) % 4], high[(i + 1) % 4], high[i]] for i in range(4)]

def platform_face(xc, yc, h):
    return [[(*c, 0.0) for c in np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]]) * PLAT / 2 @ rot_z(h)[:2, :2].T + [xc, yc]]]

fig = plt.figure(figsize=(9, 7))
axis = fig.add_subplot(projection='3d', computed_zorder=False)
allx = np.concatenate([f['x'] for f in flights.values()] + [f0['x_car']])
ally = np.concatenate([f['y'] for f in flights.values()] + [f0['y_car']])
axis.set_xlim(allx.min() - 0.6, allx.max() + 0.6); axis.set_ylim(ally.min() - 0.6, ally.max() + 0.6); axis.set_zlim(Z_GROUND, 0.9)
axis.set_box_aspect((np.ptp(axis.get_xlim()), np.ptp(axis.get_ylim()), np.ptp(axis.get_zlim())))
axis.set_xlabel('x [m]'); axis.set_ylabel('y [m]'); axis.set_zlabel('z [m]'); axis.view_init(elev=30, azim=-60)
gx, gy = np.meshgrid(axis.get_xlim(), axis.get_ylim())
axis.plot_surface(gx, gy, np.full_like(gx, Z_GROUND), color='lightgray', alpha=0.3, zorder=0)
axis.plot(f0['x_car'], f0['y_car'], np.full_like(t, Z_GROUND), '--', color='steelblue', lw=1, zorder=1)
body = Poly3DCollection(box_faces(0, 0, 0), facecolor='lightsteelblue', edgecolor='k', alpha=0.9, zorder=2)
platform = Poly3DCollection(platform_face(0, 0, 0), facecolor='orange', edgecolor='k', zorder=3)
axis.add_collection3d(body); axis.add_collection3d(platform)
ARMS_BODY = np.array([[-ARM, 0, 0], [0, ARM, 0], [ARM, 0, 0], [0, -ARM, 0]])
art = {}
for name in flights:
    c = COLORS[name]
    art[name] = dict(arms=[axis.plot([], [], [], '-', color=c, lw=3, zorder=6)[0] for _ in range(2)],
                     rotors=axis.plot([], [], [], 'o', ms=6, mfc=c, mec='k', zorder=7, label=name)[0],
                     trail=axis.plot([], [], [], ':', color=c, lw=1.2, zorder=5)[0],
                     drop=axis.plot([], [], [], '-', color=c, lw=1, alpha=0.6, zorder=4)[0])
axis.legend(loc='upper right')
label = axis.text2D(0.02, 0.95, '', transform=axis.transAxes, family='monospace')

def draw(k):
    body.set_verts(box_faces(f0['x_car'][k], f0['y_car'][k], heading[k]))
    platform.set_verts(platform_face(f0['x_car'][k], f0['y_car'][k], heading[k]))
    txt = f't = {t[k]:4.2f} s'
    for name, f in flights.items():
        a = art[name]
        M = ARMS_BODY @ rotation(f['phi'][k], f['theta'][k], f['psi'][k]).T + [f['x'][k], f['y'][k], f['z'][k]]
        for line, (i, j) in zip(a['arms'], [(0, 2), (1, 3)]):
            line.set_data_3d(M[[i, j], 0], M[[i, j], 1], M[[i, j], 2])
        a['rotors'].set_data_3d(M[:, 0], M[:, 1], M[:, 2])
        a['trail'].set_data_3d(f['x'][:k + 1], f['y'][:k + 1], f['z'][:k + 1])
        a['drop'].set_data_3d([f['x'][k]] * 2, [f['y'][k]] * 2, [f['z'][k], 0.0])
        txt += f'   {name}: {100 * f["gap"][k]:4.1f} cm'
    label.set_text(txt)
    return []

anim = animation.FuncAnimation(fig, draw, frames=n, interval=Ts * 1000)
anim.save(FIG_DIR / 'compare_rl_cascade.gif', writer=animation.PillowWriter(fps=int(round(1 / Ts))))
plt.close(fig)
print('saved:', (FIG_DIR / 'compare_rl_cascade.gif').resolve())
HTML(anim.to_jshtml())